# Streaming Market Revenue Analysis — Netflix

Analyzes Netflix's revenue drivers using public 10-K / quarterly filings, segmenting growth by region (**UCAN, EMEA, LATAM, APAC**) to identify the primary forward growth levers, size the **ad-supported tier** opportunity, and quantify the revenue impact of a churn reduction.

**Author:** Harry Vo  
**Data note:** Regional memberships and ARM are approximated from Netflix's publicly filed segment disclosures and shareholder letters. Reproducible estimates, not audited values — edit `data/generate_data.py` for scenarios.

In [1]:
import os, sys
# Anchor to the repo root so relative paths work whether the kernel starts
# in the repo root or in notebooks/.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('working dir:', os.getcwd())

working dir: /sessions/sweet-busy-meitner/mnt/outputs/netflix-streaming-revenue-analysis


## 1. Setup & data generation

In [2]:
import sys, subprocess
subprocess.run([sys.executable, 'data/generate_data.py'], check=True)
import pandas as pd
regions = pd.read_csv('data/regions.csv')
proj    = pd.read_csv('data/projections.csv')
regions

Wrote data/regions.csv and data/projections.csv


,region,members_m,arm_usd,annual_rev_b
0,UCAN,89.6,17.3,18.60
1,EMEA,96.1,11.0,12.69
2,LATAM,49.2,8.6,5.08
3,APAC,50.3,7.4,4.47


## 2. Revenue mix by region

Annual revenue = members x ARM x 12. UCAN still leads on ARM; EMEA leads on member count.

In [3]:
from src.analysis import revenue_mix
mix = revenue_mix(regions)
mix

,region,members_m,arm_usd,annual_rev_b,rev_share_pct
0,UCAN,89.6,17.3,18.60,45.5
1,EMEA,96.1,11.0,12.69,31.1
2,LATAM,49.2,8.6,5.08,12.4
3,APAC,50.3,7.4,4.47,10.9


## 3. Regional growth (CAGR 2024-2028)

Driven by member growth and ARM growth assumptions per region.

In [4]:
from src.analysis import cagr_by_region
cagr_by_region(proj)

,region,rev_2024_b,rev_2028_b,cagr_pct
0,APAC,4.47,9.86,21.9
1,EMEA,12.69,20.98,13.4
2,LATAM,5.08,7.49,10.2
3,UCAN,18.60,24.49,7.1


## 4. Projection chart

In [5]:
import matplotlib.pyplot as plt
for region, g in proj.groupby('region'):
    plt.plot(g['year'], g['annual_rev_b'], marker='o', label=region)
plt.title('Projected Netflix revenue by region (2024-2028)')
plt.ylabel('Revenue ($B)'); plt.xlabel('Year'); plt.legend(); plt.show()

## 5. Ad-supported tier opportunity

Incremental annual revenue if a share of each region's members sit on the ad-supported plan (penetration and ad ARM are editable scenario inputs).

In [6]:
from src.analysis import ad_tier_opportunity
ad = ad_tier_opportunity(regions)
print('Total incremental ad-tier revenue: ${:.2f}B/yr'.format(ad['ad_rev_b'].sum()))
ad

Total incremental ad-tier revenue: $3.68B/yr


,region,members_m,ad_members_m,ad_rev_b
0,UCAN,89.6,17.9,1.94
1,EMEA,96.1,17.3,1.25
2,LATAM,49.2,5.9,0.28
3,APAC,50.3,5.0,0.21


## 6. Churn sensitivity

In [7]:
from src.analysis import churn_impact
for pp in (0.5, 1.0, 1.5, 2.0):
    print(f'{pp}pp churn reduction -> ${churn_impact(regions, pp)}B/yr retained')

0.5pp churn reduction -> $0.2B/yr retained
1.0pp churn reduction -> $0.41B/yr retained
1.5pp churn reduction -> $0.61B/yr retained
2.0pp churn reduction -> $0.82B/yr retained


## 7. Recommendation

- **APAC** is the fastest-growing region and the primary forward growth lever, led by member growth.
- The **ad-supported tier** is the second lever — most impactful in price-sensitive regions (LATAM, APAC) where it expands the addressable base rather than cannibalizing premium ARM.
- **Prioritize ad-tier expansion in price-sensitive regions plus retention**: the ad tier and a modest churn reduction are each material to total revenue.

## 8. Full script run

In [8]:
from src import analysis
analysis.main()


=== Revenue mix by region (~FY2024) ===
region  members_m  arm_usd  annual_rev_b  rev_share_pct
  UCAN       89.6     17.3         18.60           45.5
  EMEA       96.1     11.0         12.69           31.1
 LATAM       49.2      8.6          5.08           12.4
  APAC       50.3      7.4          4.47           10.9

=== Projected growth 2024-2028 (CAGR) ===
region  rev_2024_b  rev_2028_b  cagr_pct
  APAC        4.47        9.86      21.9
  EMEA       12.69       20.98      13.4
 LATAM        5.08        7.49      10.2
  UCAN       18.60       24.49       7.1

=== Ad-tier opportunity (annual) ===
region  members_m  ad_members_m  ad_rev_b
  UCAN       89.6          17.9      1.94
  EMEA       96.1          17.3      1.25
 LATAM       49.2           5.9      0.28
  APAC       50.3           5.0      0.21
Total incremental ad-tier revenue: $3.68B/yr

=== Churn impact ===
A 1.5pp churn reduction retains ~$0.61B in annual revenue.

Saved outputs/fig_revenue_by_region.png, outputs/fig_pro